# Agent World SDK — Quick Start

This notebook demonstrates how to use the **Agent World Python SDK** to connect to a running Agent World instance,
query world state, run experiments, export data, and analyze results.

**Prerequisites:**
- Agent World running locally at `http://localhost:8080`
- `pip install agent-world-sdk pandas`
- A valid API key set in the server's `API_KEYS` environment variable

## 1. Connect to Agent World

Create a client with your API key. The key must match one of the values in the server's `API_KEYS` environment variable.

In [ ]:
from agent_world_sdk import AgentWorldClient

client = AgentWorldClient("http://localhost:8080", api_key="my-secret-key-123")
print("Connected to Agent World")

## 2. Query World State

Get the current world state — tick, agent counts, organization count, and resource distribution.

In [ ]:
state = client.world.state()
print(f"Tick: {state['tick']}")
print(f"Agents: {state['alive_count']} alive / {state['dead_count']} dead")
print(f"Organizations: {state['org_count']}")
print(f"Total money: {state['total_money']}")
print(f"Total tokens: {state['total_tokens']}")

rd = state['resource_distribution']
print(f"\nGini coefficient: {rd['gini_coefficient']}")
print(f"Avg money/agent: {rd['avg_money_per_agent']:.1f}")
print(f"Avg tokens/agent: {rd['avg_tokens_per_agent']:.1f}")

## 3. List Agents

Retrieve all agents and show their status.

In [ ]:
# List all agents
agents = client.agents.list()
print(f"Total agents: {len(agents)}")

# Show first 5
for agent in agents[:5]:
    status = "alive" if agent.get("alive") else "dead"
    print(f"  {agent.get('id', '?')}: {agent.get('name', '?')} [{status}] — tokens={agent.get('tokens', 0)}")

## 4. Deep Agent Profile

Get detailed information about a specific agent, including organization membership and reputation.

In [ ]:
# Get profile of the first agent
first_agent = agents[0]['id'] if agents else None

if first_agent:
    profile = client.agents.profile(first_agent)
    print(f"Agent: {profile['name']} (ID: {profile['id']})")
    print(f"  Phase: {profile['phase']}")
    print(f"  Tokens: {profile['tokens']}")
    print(f"  Money: {profile['money']}")
    print(f"  Survived: {profile['ticks_survived']} ticks")
    
    if profile.get('organization'):
        org = profile['organization']
        print(f"  Org: {org['org_name']} ({org['org_type']}) — role: {org['role']}")
    
    if profile.get('reputation') is not None:
        print(f"  Reputation: {profile['reputation']:.2f}")
else:
    print("No agents found")

## 5. Emergence Metrics

Get cultural diversity, organization formation, and economic concentration metrics.

In [ ]:
# Get detailed emergence metrics
emergence = client._get("/api/v2/metrics/emergence")

cd = emergence['cultural_diversity']
print("Cultural Diversity:")
print(f"  Phase distribution: {cd['phase_distribution']}")

om = emergence['organization_metrics']
print(f"\nOrganizations: {om['total_orgs']} total ({om['active_orgs']} active)")
print(f"  Type distribution: {om['org_type_distribution']}")

ec = emergence['economic_concentration']
print(f"\nEconomic Concentration:")
print(f"  Gini: {ec['gini_coefficient']}")
print(f"  Top 10% share: {ec['top_10_percent_share']}")

## 6. Run an Experiment

Create an experiment, start it, inject an event, and retrieve results.

Experiments are **recording sessions** — they capture tick snapshots at each lifecycle transition
and log any injected events, but don't isolate the world state.

In [ ]:
# Create an experiment
exp = client.experiments.create(
    description="SDK quickstart experiment",
    agent_count=10,
    target_ticks=100
)
print(f"Created experiment: {exp.id}")

# Start it
result = exp.start()
print(f"Status: {result['status']}")

In [ ]:
# Inject an event — give an agent some extra tokens
if first_agent:
    inject_result = exp.inject(
        "add_resource",
        agent_id=first_agent,
        payload={"resource": "tokens", "amount": 500}
    )
    print(f"Injection: {inject_result['status']}")

In [ ]:
# Stop the experiment
stop_result = exp.stop()
print(f"Status: {stop_result['status']}")

# Get full results
results = exp.results()
print(f"\nExperiment results:")
print(f"  Start tick: {results['start_tick']}")
print(f"  End tick: {results['end_tick']}")
print(f"  Snapshots captured: {len(results['tick_snapshots'])}")
print(f"  Injections: {len(results['injections'])}")

# Show tick snapshots
for snap in results['tick_snapshots']:
    print(f"  Tick {snap['tick']}: {snap['alive_count']} alive, {snap['total_tokens']} tokens")

## 7. Export Data

Export world state and metrics in formats ready for analysis with pandas, Gephi, or other tools.

In [ ]:
import pandas as pd
import io

# Export world state as CSV and load into pandas
world_csv = client.export.world(format="csv")
df_agents = pd.read_csv(io.StringIO(world_csv))
print(f"Exported {len(df_agents)} agents")
df_agents.head()

In [ ]:
# Export metrics time series
metrics_csv = client.export.timeseries(format="csv")
df_metrics = pd.read_csv(io.StringIO(metrics_csv))
print(f"Exported {len(df_metrics)} time-series rows")
df_metrics.head()

In [ ]:
# Export agent interaction graph
graph = client.export.graph(format="json")
print(f"Nodes: {len(graph['nodes'])}")
print(f"Edges: {len(graph['edges'])}")

# Show top interactions
edges_df = pd.DataFrame(graph['edges'])
if not edges_df.empty:
    edges_df.sort_values('weight', ascending=False).head(10)

## 8. Analyze Results

Use the SDK's built-in analysis tools for cultural diversity and trust network analysis.
These are **pure computation** — no API calls, they operate on data you've already fetched.

In [ ]:
# Cultural diversity analysis
# Build a list of agent profiles for analysis
agent_profiles = [client.agents.profile(a['id']) for a in agents[:10]]

diversity = client.analyze.cultural_diversity(agent_profiles)
print("Cultural Diversity Analysis:")
print(f"  Shannon entropy: {diversity['shannon_entropy']}")
print(f"  Unique phases: {diversity['unique_phases']}")
print(f"  Simpson index: {diversity['simpson_index']}")
print(f"  Phase counts: {diversity['phase_counts']}")

In [ ]:
# Trust network analysis
trust = client.analyze.trust_network(graph['edges'])
print("Trust Network Analysis:")
print(f"  Nodes: {trust['node_count']}")
print(f"  Edges: {trust['edge_count']}")
print(f"  Avg degree: {trust['avg_degree']}")
print(f"  Density: {trust['density']}")

## Summary

You've learned how to:

1. **Connect** to Agent World with an API key
2. **Query** world state and agent profiles
3. **Run experiments** — create, start, inject, stop, retrieve results
4. **Export data** as JSON, CSV, or GraphML for external tools
5. **Analyze** cultural diversity and trust networks

Next steps:
- Explore the full [API Reference](../docs/api-reference.md) for all v2 endpoints
- Try different injection types to see how the simulation responds
- Use the GraphML export with [Gephi](https://gephi.org/) for network visualization